In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import accuracy_score

1. В данной работе проверьте целесообразность каждого необязательного преобразования данных путем проверки, увеличивает ли данное преобразование точность модели. Проверьте на простом виде модели (логит регрессия, дерево решений или случайный лес).

In [ ]:
prsa_data = pd.read_csv("https://raw.githubusercontent.com/koroteevmv/ML_course/refs/heads/main/ML5.2%20numeric%20features/data/PRSA_Data.csv")
prsa_data[prsa_data == -1] = np.nan

# кодирование целевой переменной
le = LabelEncoder()
y = le.fit_transform(prsa_data['AQI Label'])

# базовые признаки
X_base = prsa_data[['SO2', 'NO2', 'CO', 'O3', 'PRES', 'WSPM']].copy()
X_base = X_base.fillna(X_base.median())

# модель на исходных данных
X_train, X_test, y_train, y_test = train_test_split(X_base, y, test_size=0.2, random_state=42)
rf = RandomForestClassifier(n_estimators=50, random_state=42)
rf.fit(X_train, y_train)
base_acc = accuracy_score(y_test, rf.predict(X_test))

print(f"Точность на исходных данных: {base_acc:.4f}")

# проверка преобразований
# 1.бинаризация RAIN
X_binarized = X_base.copy()
X_binarized['IS_RAIN'] = (prsa_data['RAIN'] > 0).astype(int)
X_binarized = X_binarized.drop(columns=['RAIN'], errors='ignore')
X_binarized = X_binarized.fillna(X_binarized.median())

X_train, X_test, y_train, y_test = train_test_split(X_binarized, y, test_size=0.2, random_state=42)
rf.fit(X_train, y_train)
bin_acc = accuracy_score(y_test, rf.predict(X_test))
print(f"Бинаризация RAIN: {bin_acc:.4f} (изменение: {bin_acc - base_acc:.4f})")

# 2.группировка CO
X_grouped = X_base.copy()
X_grouped['CO_group'] = pd.cut(X_grouped['CO'], bins=[0, 250, 320, 10000], labels=[1, 2, 3])
X_grouped['CO_group'] = X_grouped['CO_group'].cat.add_categories([0])
X_grouped['CO_group'] = X_grouped['CO_group'].fillna(0).astype(int)
X_grouped = X_grouped.drop(columns=['CO'])
X_grouped = X_grouped.fillna(X_grouped.median())

X_train, X_test, y_train, y_test = train_test_split(X_grouped, y, test_size=0.2, random_state=42)
rf.fit(X_train, y_train)
group_acc = accuracy_score(y_test, rf.predict(X_test))
print(f"Группировка CO: {group_acc:.4f} (изменение: {group_acc - base_acc:.4f})")

# 3.логарифмирование SO2
X_log = X_base.copy()
X_log['SO2_log'] = np.log(X_log['SO2'] + 1)
X_log = X_log.drop(columns=['SO2'])
X_log = X_log.fillna(X_log.median())

X_train, X_test, y_train, y_test = train_test_split(X_log, y, test_size=0.2, random_state=42)
rf.fit(X_train, y_train)
log_acc = accuracy_score(y_test, rf.predict(X_test))
print(f"Логарифмирование SO2: {log_acc:.4f} (изменение: {log_acc - base_acc:.4f})")

Точность на исходных данных: 0.9689
Бинаризация RAIN: 0.9676 (изменение: -0.0013)
Группировка CO: 0.9487 (изменение: -0.0202)
Логарифмирование SO2: 0.9673 (изменение: -0.0016)


2. Проанализируйте эффективность использования различных методов заполнения пропущенных значений.

In [ ]:
#сравниваем на столбце O3
X_o3 = prsa_data[['O3', 'SO2', 'NO2', 'CO', 'PRES', 'WSPM', 'AQI Label']].copy()
X_o3 = X_o3.dropna(subset=['AQI Label'])
y_o3 = le.fit_transform(X_o3['AQI Label'])
X_o3 = X_o3.drop(columns=['AQI Label'])

methods = {
    'median': X_o3.fillna(X_o3.median()),
    'mean': X_o3.fillna(X_o3.mean()),
    'zero': X_o3.fillna(0),
    'ffill': X_o3.fillna(method='ffill')
}

print("\nСравнение методов заполнения пропусков:")
for name, X_filled in methods.items():
    X_train, X_test, y_train, y_test = train_test_split(X_filled, y_o3, test_size=0.2, random_state=42)
    rf.fit(X_train, y_train)
    acc = accuracy_score(y_test, rf.predict(X_test))
    print(f"  {name}: {acc:.4f}")

/tmp/ipykernel_2667/1430512749.py:11: FutureWarning: DataFrame.fillna with 'method' is deprecated and will raise in a future version. Use obj.ffill() or obj.bfill() instead.
  'ffill': X_o3.fillna(method='ffill')



Сравнение методов заполнения пропусков:
  median: 0.9682
  mean: 0.9769
  zero: 0.9775
  ffill: 0.9695


3. Попробуйте придумать другие методы инжиниринга признаков в этом датасете. Оцените их эффект по какой-то модели.

In [ ]:
X_new = X_base.copy()

#признак 1: отношение NO2 к SO2 (соотношение загрязнителей)
X_new['NO2_SO2_ratio'] = X_new['NO2'] / (X_new['SO2'] + 1)

#признак 2: произведение CO и WSPM (ветер разносит угарный газ)
X_new['CO_WSPM'] = X_new['CO'] * X_new['WSPM']

#признак 3: разница PRES от нормального давления (1013.25 гПа)
X_new['PRES_diff'] = X_new['PRES'] - 1013.25

#заполнение пропусков
X_new = X_new.fillna(X_new.median())

X_train, X_test, y_train, y_test = train_test_split(X_new, y, test_size=0.2, random_state=42)
rf.fit(X_train, y_train)
new_acc = accuracy_score(y_test, rf.predict(X_test))
print(f"\nТочность с новыми признаками: {new_acc:.4f}")
print(f"Улучшение относительно базовой: {new_acc - base_acc:.4f}")


Точность с новыми признаками: 0.9673
Улучшение относительно базовой: -0.0016


4. Перед началом обработки данных разбейте датасет на тестовую и обучающую выборки. Очистите по методу из работы обучающую выборку. Повторите обработку на тестовой выборке. При этом позаботьтесь, чтобы все параметрические преобразования (клиппинг, нормализация, группировка и так далее).

In [ ]:
prsa_data = pd.read_csv("https://raw.githubusercontent.com/koroteevmv/ML_course/refs/heads/main/ML5.2%20numeric%20features/data/PRSA_Data.csv")
prsa_data[prsa_data == -1] = np.nan

#разделение
train_data, test_data = train_test_split(prsa_data, test_size=0.2, random_state=42)

#параметры для преобразований
median_so2 = train_data['SO2'].median()
median_no2 = train_data['NO2'].median()
co_bins = [0, 250, 320, 10000]

#обработка train
train_data['SO2'] = train_data['SO2'].fillna(median_so2)
train_data['NO2'] = train_data['NO2'].fillna(median_no2)
train_data['IS_RAIN'] = (train_data['RAIN'] > 0).astype(int)
train_data['CO_group'] = pd.cut(train_data['CO'], bins=co_bins, labels=[1, 2, 3])
train_data['CO_group'] = train_data['CO_group'].cat.add_categories([0])
train_data['CO_group'] = train_data['CO_group'].fillna(0).astype(int)

#обработка test
test_data['SO2'] = test_data['SO2'].fillna(median_so2)
test_data['NO2'] = test_data['NO2'].fillna(median_no2)
test_data['IS_RAIN'] = (test_data['RAIN'] > 0).astype(int)
test_data['CO_group'] = pd.cut(test_data['CO'], bins=co_bins, labels=[1, 2, 3])
test_data['CO_group'] = test_data['CO_group'].cat.add_categories([0])
test_data['CO_group'] = test_data['CO_group'].fillna(0).astype(int)

print(f"\nОбучающая выборка: {len(train_data)} строк")
print(f"Тестовая выборка: {len(test_data)} строк")
print(f"Распределение CO_group в train:\n{train_data['CO_group'].value_counts().sort_index()}")


Обучающая выборка: 28051 строк
Тестовая выборка: 7013 строк
Распределение CO_group в train:
CO_group
0     1444
1     1362
2     1927
3    23318
Name: count, dtype: int64


5. Создайте воспроизводимый код обработки данного датасета.

In [ ]:
def preprocess_prsa_data(df, fit_mode=True, fit_params=None):
    df = df.copy()
    df[df == -1] = np.nan

    if fit_mode:
        #обучение параметров
        fit_params = {
            'so2_median': df['SO2'].median(),
            'no2_median': df['NO2'].median(),
            'co_median': df['CO'].median(),
            'o3_median': df['O3'].median(),
            'pres_bounds': (992, 1034)
        }

    #заполнение пропусков
    df['SO2'] = df['SO2'].fillna(fit_params['so2_median'])
    df['NO2'] = df['NO2'].fillna(fit_params['no2_median'])
    df['CO'] = df['CO'].fillna(fit_params['co_median'])
    df['O3'] = df['O3'].fillna(fit_params['o3_median'])

    #бинаризация
    df['IS_RAIN'] = (df['RAIN'] > 0).astype(int)

    #ограничение больших значений
    lower, upper = fit_params['pres_bounds']
    df['PRES'] = df['PRES'].clip(lower, upper)

    #группировка CO
    df['CO_group'] = pd.cut(df['CO'], bins=[0, 250, 320, 10000], labels=[1, 2, 3])
    df['CO_group'] = df['CO_group'].cat.add_categories([0])
    df['CO_group'] = df['CO_group'].fillna(0).astype(int)

    #логарифмирование
    df['SO2_log'] = np.log(df['SO2'] + 1)

    #удаление лишних колонок
    cols_to_drop = [col for col in ['RAIN', 'No', 'wd'] if col in df.columns]
    df = df.drop(columns=cols_to_drop, errors='ignore')

    return df, fit_params

print(f"Форма train: {train_processed.shape}, test: {test_processed.shape}")
print(f"Колонки: {list(train_processed.columns)}")

Форма train: (28051, 11), test: (7013, 11)
Колонки: ['Unnamed: 0', 'SO2', 'NO2', 'CO', 'O3', 'PRES', 'WSPM', 'AQI Label', 'IS_RAIN', 'CO_group', 'SO2_log']


7. Повторите обработку численных параметров в датасете "Титаник".

In [ ]:
titanic = pd.read_csv("https://raw.githubusercontent.com/koroteevmv/ML_course/refs/heads/main/ML5.2%20numeric%20features/data/titanic.csv")
print("Исходные данные:")
print(titanic.info())
print(titanic.head(5))

#обработка численных параметров Титаника
def process_titanic_numeric(df):
    df = df.copy()

    #заполнение пропусков в возрасте/в Fare
    df['Age'] = df.groupby('Pclass')['Age'].transform(lambda x: x.fillna(x.median()))
    df['Fare'] = df['Fare'].fillna(df['Fare'].median())

    #бинаризация (братья/супруги на борту)
    df['has_sibsp'] = (df['SibSp'] > 0).astype(int)
    #бинаризация (родители/дети на борту)
    df['has_parch'] = (df['Parch'] > 0).astype(int)

    #группировка возраста
    df['Age_group'] = pd.cut(df['Age'], bins=[0, 12, 18, 35, 60, 100],
                             labels=['child', 'teen', 'adult', 'middle', 'senior'])

    #логарифмирование стоимости билета
    df['Fare_log'] = np.log(df['Fare'] + 1)

    #группировка стоимости билета
    df['Fare_group'] = pd.qcut(df['Fare'], q=4, labels=['low', 'medium', 'high', 'very_high'])

    #создание семейного признака
    df['Family_size'] = df['SibSp'] + df['Parch'] + 1
    df['is_alone'] = (df['Family_size'] == 1).astype(int)

    return df

#обработка
titanic_processed = process_titanic_numeric(titanic)

print("\nПосле обработки:")
print(titanic_processed[['Age', 'SibSp', 'Parch', 'Fare', 'Family_size', 'is_alone',
                          'Age_group', 'Fare_group', 'Fare_log']].head(10))

features = ['Pclass', 'Age', 'SibSp', 'Parch', 'Fare', 'Family_size', 'is_alone', 'Fare_log']
X = titanic_processed[features].copy()
y = titanic_processed['Survived']

#заполняем оставшиеся пропуски, если есть
X = X.fillna(X.median())

#разделение и обучение
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
rf = RandomForestClassifier(n_estimators=100, random_state=42)
rf.fit(X_train, y_train)
accuracy = accuracy_score(y_test, rf.predict(X_test))

print(f"\nТочность модели Random Forest: {accuracy:.4f}")
print(f"Важность признаков:")
for feat, imp in zip(features, rf.feature_importances_):
    print(f"  {feat}: {imp:.4f}")

Исходные данные:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 891 entries, 0 to 890
Data columns (total 12 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   PassengerId  891 non-null    int64  
 1   Survived     891 non-null    int64  
 2   Pclass       891 non-null    int64  
 3   Name         891 non-null    object 
 4   Sex          891 non-null    object 
 5   Age          714 non-null    float64
 6   SibSp        891 non-null    int64  
 7   Parch        891 non-null    int64  
 8   Ticket       891 non-null    object 
 9   Fare         891 non-null    float64
 10  Cabin        204 non-null    object 
 11  Embarked     889 non-null    object 
dtypes: float64(2), int64(5), object(5)
memory usage: 83.7+ KB
None
   PassengerId  Survived  Pclass  \
0            1         0       3   
1            2         1       1   
2            3         1       3   
3            4         1       1   
4            5         0       3   

     

8. (*) Создайте код, реализующий алгоритм очистки данных автоматически (для любого датасета).

In [ ]:
def auto_clean_data(df, threshold=3):
    df = df.copy()

    #замена специальных значений на NaN
    for col in df.select_dtypes(include=[np.number]).columns:
        mean, std = df[col].mean(), df[col].std()
        outliers = (df[col] < mean - threshold * std) | (df[col] > mean + threshold * std)
        if outliers.sum() > 0 and outliers.sum() < len(df) * 0.05:
            df.loc[outliers, col] = np.nan

    #замена отрицательных значений на NaN
    for col in df.select_dtypes(include=[np.number]).columns:
        if df[col].min() < 0 and df[col].mean() > 0:
            df.loc[df[col] < 0, col] = np.nan

    #заполнение пропусков
    for col in df.select_dtypes(include=[np.number]).columns:
        if df[col].isnull().sum() > 0:
            if df[col].skew() > 1:
                df[col] = df[col].fillna(df[col].median())
            else:
                df[col] = df[col].fillna(df[col].mean())
    for col in df.select_dtypes(include=['object']).columns:
        df[col] = df[col].fillna(df[col].mode()[0] if len(df[col].mode()) > 0 else 'unknown')

    #округление чисел
    for col in df.select_dtypes(include=[np.number]).columns:
        if df[col].dtype == float and df[col].nunique() > 100:
            decimals = max(0, 3 - int(np.log10(df[col].max())))
            df[col] = df[col].round(decimals)

    return df

#тестим
prsa_clean = auto_clean_data(prsa_data)
print(f"\nАвтоматическая очистка готова. Из: {prsa_clean.shape}")
print(f"Оставшиеся пропуски: {prsa_clean.isnull().sum().sum()}")


Автоматическая очистка готова. Из: (35064, 11)
Оставшиеся пропуски: 0
